In [ ]:
from IPython.display import display, HTML, Markdown
import student_info
display(Markdown("## 商务数据理论与方法——实验1"))
display(HTML(f"<b>姓名：{student_info.NAME}&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;学号：{student_info.STUDENT_ID}</b>"))



>实验主题：描述性统计分析<br><br>
>实验目的：
>- 掌握pandas的基本用法
>- 基于pandas完成描述性分析
    * 基础统计
    * 集中趋势分析 
    * 离散程度分析
    
<font color="red">请发送至：<b>teaching\_jin_2\@126.com</b> ，邮件标题和附件标题均为：<b>商务数据分析理论与方法_学号\_姓名\_实验1</b></font>


### 数据说明
数据集包含了抽样出来的1W用户在一个月时间（11.18~12.18）之内的淘宝移动端行为数据


|字段|字段说明|提取说明|
|----------|:-------------:|------------------:|
|user_id|用户标识|抽样&字段脱敏|
|item_id|商品标识|字段脱敏|
|behavior_name|用户对商品的行为类型| 包括浏览、收藏、加购物车、购买|
|item_category|商品分类标识|字段脱敏|
|time|行为时间|精确到小时级别|

### 任务描述
基于上述数据集，完成如下的商业分析任务，通过描述性统计分析，充分挖掘数据的价值，让数据更好地为业务服务：

- 统计每日PV和UV
- 统计销量最高的前100名商品信息，包括商品标识和销售量
- 漏斗分析：用户“浏览-收藏-加购-购买”的转化率是怎样的？哪一步的折损比例最大？
- 任选一件商品，统计该商品的每天（或某天的24个小时）的PV/UC
- 任选一件商品，统计该商品的每小时/每天销量，对该商品的销量进行分布分析
- 更多有意义的问题欢迎大家发挥聪明才智来挖掘。

In [ ]:
import pandas as pd

### 1. 数据加载与基本信息查看

In [ ]:
#读取数据
data = pd.read_csv("user_action.csv",usecols=[1,2,3,4,5],encoding="utf-8")

In [ ]:
#数据备份
dt = data.copy()

In [ ]:
#数据抽样查看
dt.sample(10)

In [ ]:
#数据信息查看
dt.info()

In [ ]:
# 查看整体数据规模，包括整体数据、用户数、商品数、商品类别数等
summary={
    "整体数据":dt.shape[0],
    "用户数":len(set(dt["user_id"])),
    "商品数":len(set((dt["item_id"]))),
    "商品类别":len(set(dt["item_category"]))
}

summary

### 2. 数据处理

> 由于数据的最小统计时间单位是小时，后续的分析可以从小时、天等时间跨度进行数据分析，考虑将时间进行分割处理

In [ ]:
# 先将行为时间time分割天(date)和小时(hour)，以便后续的进一步分析
dt["date"] = dt["time"].map(lambda x:x.split(" ")[0])
dt["hour"] = dt["time"].map(lambda x:x.split(" ")[1])
dt.sample(10)

In [ ]:
# 查看数据情况
dt.info()

In [ ]:
# 数据类型转换
dt['date'] = pd.to_datetime(dt['date'])
dt['hour'] = dt['hour'].astype('int64')

In [ ]:
# 查看数据类型
dt.dtypes

### 3 数据分析

#### 3.1 流量分析
> 使用PV（页面访问量）和UV（独立访客数）指标，分析访问量和访问用户的趋势和规律。用可视化工具绘制相应的图表更好地展示数据，进而找出高峰期及用户访问偏好。同时，我们发现提供的数据有“双十二”这一特殊日期，因此可针对12月12日这一天进行具体分析，以便更好地规划营销活动。

* PV（页面访问量）即Page View，用户每次对网站的访问均被记录，用户对同一页面的多次访问，是访问量的累计。
* UV（独立访客数）即Unique Visitor，访问网站的一台电脑客户端为一个访客，根据IP地址来区分访客数。

##### 3.1.1 PV统计分析

（1）PV统计

In [ ]:
#### PV 统计
pv_daily = dt.groupby("date")['user_id'].count().reset_index().rename(columns={'user_id':'pv_daily'})
pv_daily

（2）PV排名前10的日期及PV

In [ ]:
top_10 = pv_daily.sort_values(by="pv_daily",ascending=False).head(10)
top_10

（3）PV排名后10的日期及PV

In [ ]:
last_10 = pv_daily.sort_values(by="pv_daily",ascending=True).head(10)
last_10

（4）PV描述性统计分析

In [ ]:
#pv的描述性统计分析
pv_daily["pv_daily"].describe().to_frame()

##### 3.1.2 UV统计分析
（1）UV统计

In [ ]:
#### UV统计
uv_daily = dt.groupby("date")['user_id'].apply(lambda x:len(set(x.unique()))).reset_index().rename(columns={"user_id":"uv_daily"})
uv_daily

（2）UV的描述性统计分析

In [ ]:
### UV的描述性统计分析
uv_daily["uv_daily"].describe().to_frame()

(3)UV与PV的对比分析

In [ ]:
import matplotlib.pyplot as plt
#将输出图形嵌入到网页中
%matplotlib inline
# 用折线图可视化展示30天的PV和UV数据
fig = plt.figure(figsize=(15,10))
ax1 =fig.add_subplot(2,1,1)
pv_daily.plot(x='date', y='pv_daily', ax=ax1)
ax2 =fig.add_subplot(2,1,2)
uv_daily.plot(x='date', y='uv_daily', ax=ax2)
ax1.set_title('pv_daily')
ax2.set_title('uv_daily')

#### 3.1.3 每小时PV和UV数据分析

> 分析每小时的PV和UV数据，探索用户在一天当中对淘宝的使用的规律和趋势

In [ ]:
# show me your code

pv_hourly = dt.groupby("hour")["user_id"].count().reset_index().rename(columns={"user_id": "pv"})
uv_hourly = dt.groupby("hour")["user_id"].nunique().reset_index().rename(columns={"user_id": "uv"})

fig = plt.figure(figsize=(15,5))
ax1 = fig.add_subplot(1,1,1)
ax1.plot(pv_hourly["hour"], pv_hourly["pv"], label="PV")
ax2 = ax1.twinx()
ax2.plot(uv_hourly["hour"], uv_hourly["uv"], color="r", label="UV")
fig.legend()
ax1.set_ylabel("PV")
ax2.set_ylabel("UV")
ax1.set_xlabel("Hour")
ax1.set_title("PV, UV Hourly")


#### 3.1.4 特殊日期（双十二）的每小时PV和UV数据分析

> 针对双十二这类大型促销活动，进一步进行每小时的PV和UV数据分析

In [ ]:
# show me your code

dt12 = dt[dt['date'] == '2014-12-12']

pv_hourly12 = dt12.groupby('hour')['user_id'].count().reset_index().rename(columns={'user_id': 'pv'})
uv_hourly12 = dt12.groupby('hour')['user_id'].nunique().reset_index().rename(columns={'user_id': 'uv'})

fig = plt.figure(figsize=(15,5))
ax1 = fig.add_subplot(1,1,1)
ax1.plot(pv_hourly12["hour"], pv_hourly["pv"], label="PV")
ax2 = ax1.twinx()
ax2.plot(uv_hourly12["hour"], uv_hourly["uv"], color="r", label="UV")
fig.legend()
ax1.set_ylabel("PV")
ax2.set_ylabel("UV")
ax1.set_xlabel("Hour")
ax1.set_title("1212 PV, UV Hourly")

#### 3.2 销量分析
>  统计销量最高的前100名商品信息，包括商品标识和销售量

In [ ]:
# show me you code

sales = dt[dt['behavior_name']=='购买'].groupby('item_id').count()['behavior_name'].reset_index().rename(columns={'behavior_name':'buys'}).sort_values('buys', ascending=False)
sales.head(100)

#### 3.3 漏斗分析

> 漏斗分析是指用户在进行某个特定操作时，从最初的浏览到最终的购买等行为之间的流程和转化率的分析。它可以帮助商家了解用户在购买过程中的转化情况，以便更好地优化营销策略。在本次分析中，我们将通过对用户的浏览、收藏、加购物车、购买等行为进行漏斗分析，以了解用户在不同阶段的行为转化率和转化情况，找出漏斗中的瓶颈点和优化方向。

（1）一个月转化情况分析

In [ ]:
# 获取用户行为数据
behavior_statc = dt.groupby(['behavior_name'])['user_id'].count()
behavior_statc

In [ ]:
# 提取用户在电商平台上的浏览、收藏、加购、购买等行为
browse = behavior_statc.iloc[2]   # 浏览
collect = behavior_statc.iloc[1]  # 收藏
add_cart = behavior_statc.iloc[0] # 加购
buy = behavior_statc.iloc[3]      # 购买

# 计算转化率
collect_rate = round(collect / browse * 100, 2)    # 收藏率
add_cart_rate = round(add_cart / browse * 100, 2) # 加购率
buy_rate = round(buy / add_cart * 100, 2)          # 购买率

print('收藏率:', collect_rate, '%')
print('加购物车率:', add_cart_rate, '%')
print('购买率:', buy_rate, '%')

（2）“双十二”当天转化情况分析

In [ ]:
# show me your code

bs12 = dt12.groupby(['behavior_name'])['user_id'].count()
browse12 = bs12.iloc[2]
collect12 = bs12.iloc[1]
add_cart12 = bs12.iloc[0]
buy12 = bs12.iloc[3]

collect_rate12 = round(collect12 / browse12 * 100, 2)    # 收藏率
add_cart_rate12 = round(add_cart12 / browse12 * 100, 2) # 加购率
buy_rate12 = round(buy12 / add_cart12 * 100, 2)          # 购买率

print('收藏率:', collect_rate12, '%')
print('加购物车率:', add_cart_rate12, '%')
print('购买率:', buy_rate12, '%')

#### 3.4 指定产品进行描述性统计分析

> 对于给定产品（item_id），以日或者小时为单位，分析产品的UV/PV，转化情况等

In [ ]:
# show me your code

id = int(input())
dt_i = dt[dt['item_id']==id]

# 日pv uv
pv_daily_i = dt_i.groupby('date')['user_id'].count().reset_index().rename(columns={'user_id': 'pv'})
uv_daily_i = dt_i.groupby('date')['user_id'].nunique().reset_index().rename(columns={'user_id': 'uv'})

# 日转化率
bs_daily = dt_i.groupby(['date', 'behavior_name'])['user_id'].count().reset_index().pivot(index='date', columns='behavior_name', values='user_id').reset_index()
bs_daily['收藏率'] = bs_daily['收藏'] / bs_daily['浏览']
bs_daily['加购物车率'] = bs_daily['加购物车'] / bs_daily['浏览']
bs_daily['购买率'] = bs_daily['购买'] / bs_daily['加购物车']
bs_daily = bs_daily.fillna(0)

# 图
fig3 = plt.figure(figsize=(15,5))

ax1 = fig3.add_subplot(1,1,1)
ax2 = ax1.twinx()

pv_daily_i.plot(x='date', y='pv', ax=ax1, label='PV', legend=False)
uv_daily_i.plot(x='date', y='uv', ax=ax2, label='UV', color='r', legend=False)

ax1.set_ylabel('PV')
ax2.set_ylabel('UV')
ax1.set_xlabel('Date')
ax1.set_title(f'Daily PV, UV of {id}')
fig3.legend()

fig4 = plt.figure(figsize=(15,5))
ax3 = fig4.add_subplot(1,1,1)
bs_daily.plot(x='date', y='收藏率', ax=ax3, label='Collect rate', color='b', legend=False)
bs_daily.plot(x='date', y='加购物车率', ax=ax3, label='Add cart rate', color='y', legend=False)
ax4 = ax3.twinx()
bs_daily.plot(x='date', y='购买率', ax=ax4, label='Buy rate', color='g', legend=False)

ax3.set_xlabel('Date')
ax3.set_ylabel('Collect, Add Cart Rate')
ax4.set_ylabel('Buy Rate')
ax3.set_title(f'Daily Collect, Add Cart, Buy Rate of {id}')
fig4.legend()

### Enjoy